In [ ]:
# Reaproveitando a célula de conexão do 01_coleta (engine)

def ler(tabela):
    return pd.read_sql(f"SELECT * FROM {tabela}", engine)

orders    = ler("orders")
items     = ler("order_items")
reviews   = ler("order_reviews")
products  = ler("products")
customers = ler("customers")
geo       = ler("geolocation")
trans     = ler("product_category_translation")

In [ ]:
# Diagnóstico Geral (Nulos, duplicados, tipos)

def diagnostico(df, nome):
    print(f"\n=== {nome} {df.shape} ===")
    resumo = pd.DataFrame({
        "nulos": df.isna().sum(),
        "% nulos": (df.isna().mean() * 100).round(2),
        "tipo": df.dtypes,
    })
    print(resumo[resumo["nulos"] > 0])
    print("Linhas duplicadas:", df.duplicated().sum())

for nome, df in {"orders": orders, "items": items, "reviews": reviews,
                 "products": products, "customers": customers,
                 "geolocation": geo}.items():
    diagnostico(df, nome)


=== orders (99441, 8) ===
                               nulos  % nulos            tipo
order_approved_at                160     0.16  datetime64[us]
order_delivered_carrier_date    1783     1.79  datetime64[us]
order_delivered_customer_date   2965     2.98  datetime64[us]
Linhas duplicadas: 0

=== items (112650, 7) ===
Empty DataFrame
Columns: [nulos, % nulos, tipo]
Index: []
Linhas duplicadas: 0

=== reviews (99224, 7) ===
                        nulos  % nulos tipo
review_comment_title    87656    88.34  str
review_comment_message  58247    58.70  str
Linhas duplicadas: 0

=== products (32951, 9) ===
                            nulos  % nulos     tipo
product_category_name         610     1.85      str
product_name_lenght           610     1.85  float64
product_description_lenght    610     1.85  float64
product_photos_qty            610     1.85  float64
product_weight_g                2     0.01  float64
product_length_cm               2     0.01  float64
product_height_cm       

In [ ]:
# Status dos pedidos e datas de entrega

print(orders["order_status"].value_counts())

# pedidos "delivered" sem data de entrega?
entregues = orders[orders["order_status"] == "delivered"]
print("Entregues sem data de entrega:", entregues["order_delivered_customer_date"].isna().sum())

# entrega antes da compra (incoerência)?
print("Entrega antes da compra:",
      (orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]).sum())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
Entregues sem data de entrega: 8
Entrega antes da compra: 0


In [ ]:
# Avaliações por pedido

print("Pedidos com mais de 1 avaliação:",
      reviews["order_id"].duplicated().sum())

Pedidos com mais de 1 avaliação: 551


In [ ]:
# Granularidade dos itens

print("Itens por pedido (média):", round(len(items) / items["order_id"].nunique(), 2))
print("Preços <= 0:", (items["price"] <= 0).sum())
print("Frete negativo:", (items["freight_value"] < 0).sum())

Itens por pedido (média): 1.14
Preços <= 0: 0
Frete negativo: 0


In [ ]:
# Produtos e categorias

print("Produtos sem categoria:", products["product_category_name"].isna().sum())

sem_traducao = set(products["product_category_name"].dropna()) - set(trans["product_category_name"])
print("Categorias sem tradução:", sem_traducao)

Produtos sem categoria: 610
Categorias sem tradução: {'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}


In [ ]:
# Geolocalização

print("Linhas:", len(geo), "| CEPs únicos:", geo["geolocation_zip_code_prefix"].nunique())

Linhas: 1000163 | CEPs únicos: 19015


In [ ]:
# Clientes

print("customer_id únicos:", customers["customer_id"].nunique())
print("customer_unique_id únicos:", customers["customer_unique_id"].nunique())

customer_id únicos: 99441
customer_unique_id únicos: 96096


In [ ]:
# Outliers no tempo de entrega

e = entregues.dropna(subset=["order_delivered_customer_date"]).copy()
e["dias_entrega"] = (e["order_delivered_customer_date"] - e["order_purchase_timestamp"]).dt.days
print(e["dias_entrega"].describe())

count    96470.000000
mean        12.093604
std          9.551380
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: dias_entrega, dtype: float64


In [18]:
v = pd.read_sql("SELECT * FROM vw_pedidos_entregues", engine)
print(v.shape)
print("order_id duplicado:", v["order_id"].duplicated().sum())   # deve ser 0
print(v.isna().sum())

(96470, 10)
order_id duplicado: 0
order_id                           0
customer_unique_id                 0
customer_state                     0
order_purchase_timestamp           0
order_delivered_customer_date      0
order_estimated_delivery_date      0
dias_entrega                       0
dias_atraso                        0
atrasado                           0
review_score                     646
dtype: int64
